# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)

# Print a summary
dataset_metadata = dataset.metadata
print(f"{dataset_metadata.name}: {dataset_metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all available RecordSets and their fields
print("Available Record Sets:")
record_sets = dataset.record_sets
for rs in record_sets:
    print(f"@id: {rs['@id']}, name: {rs.get('name', '[No name]')}")
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    print("  Fields (@id):")
    for field in fields:
        field_id = field['@id'] if isinstance(field, dict) else field
        print(f"    - {field_id}")
    print("-")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from each record set into DataFrames
# First, get all record set @id's
record_set_ids = [rs['@id'] for rs in record_sets]
print("Record set @ids found:", record_set_ids)
dataframes = {}
for record_set_id in record_set_ids:
    try:
        print(f"Loading records for: {record_set_id}")
        records = list(dataset.records(record_set=record_set_id))
        if len(records) > 0:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"{record_set_id}:", df.shape)
            print(f"Columns (using @id):", df.columns.tolist())
            display(df.head(2))
        else:
            print(f"No records found for {record_set_id}.")
    except Exception as e:
        print(f"Failed loading {record_set_id} due to: {e}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# For EDA, pick the primary record set - you can adjust this to match your dataset after viewing its overview.
# This notebook uses a typical protein table record set @id if it exists (replace as needed after running overview)
import numpy as np

# Example - Set to your protein record set's @id (update as needed, example: 'cr:ProteinTable')
primary_record_set_id = record_set_ids[0]

df = dataframes.get(primary_record_set_id)
print(f'Using primary record set: {primary_record_set_id}')

# Display available columns (@id)
print("Columns:", df.columns.tolist())

# Try to select a numeric field (by guessing common patterns)
numeric_candidates = [c for c in df.columns if c.lower().endswith(('abundance','mw','weight','count','peptide','coverage','score')) or df[c].dtype.kind in 'fi']
if len(numeric_candidates) == 0:
    # fallback: use the first float/integer column
    for c in df.columns:
        try:
            if df[c].dropna().astype(float).apply(lambda x: np.isfinite(x)).all():
                numeric_candidates.append(c)
                break
        except:
            continue
if len(numeric_candidates) == 0:
    print("No numeric fields found.")
else:
    numeric_field = numeric_candidates[0]

    print(f"Selected numeric field for filtering: {numeric_field}")
    # Try to guess a reasonable threshold; use median
    threshold = np.nanmedian(pd.to_numeric(df[numeric_field], errors='coerce'))
    filtered_df = df[pd.to_numeric(df[numeric_field], errors='coerce') > threshold]
    print(f"Filtered records with {numeric_field} > {threshold} (median):")
    display(filtered_df.head(5))

    # Normalization
    col_norm = f"{numeric_field}_normalized"
    filtered_df[col_norm] = (pd.to_numeric(filtered_df[numeric_field], errors='coerce') - pd.to_numeric(filtered_df[numeric_field], errors='coerce').mean()) / pd.to_numeric(filtered_df[numeric_field], errors='coerce').std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, col_norm]].head(5))

    # Try grouping by another field - try 'cr:description' or similar if it exists
    group_candidates = [c for c in df.columns if 'group' in c.lower() or 'type' in c.lower() or 'description' in c.lower()]
    if len(group_candidates) > 0:
        group_field = group_candidates[0]
        print(f"Grouping by field: {group_field}")
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        display(grouped_df.head())
    else:
        group_field = None
        print('No suitable categorical field found to group by.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if 'filtered_df' in locals() and len(filtered_df) > 0 and 'numeric_field' in locals():
    # Histogram of the numeric field
    plt.figure(figsize=(7,4))
    pd.to_numeric(filtered_df[numeric_field], errors='coerce').hist(bins=30)
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.title(f'Distribution of {numeric_field}')
    plt.show()

    # If we have a group field, plot boxplot
    if 'group_field' in locals() and group_field is not None:
        plt.figure(figsize=(10,5))
        sns.boxplot(data=filtered_df, x=group_field, y=numeric_field)
        plt.title(f'{numeric_field} by {group_field}')
        plt.tight_layout()
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The dataset contains records related to protein analysis from extracellular vesicles using mass spectrometry.
- We explored available record sets and extracted data for inspection.
- Common data exploration steps such as filtering, normalization, and grouping were demonstrated.
- Visualizations show distributions of measured or computed fields, which can help in downstream analysis like biomarker discovery.
- For more advanced analysis, refer to the dataset's fields and add more domain-specific processing.